# Live Tracing and Visualizer GUI

Graphyco provides multiple ways to visualize computational graphs and monitor gradient flow:
1. **Interactive In-Notebook Visualization**: Render the interactive graph and telemetry dashboard directly inside Jupyter notebook cells with `visualize_inline()`.
2. **Live Diagnostics Monitoring**: Hook `LiveTrainingDiagnostics` directly into active training loops to track gradient health, dead neurons, and bottlenecks in real-time.
3. **Native Desktop GUI**: Launch a standalone PySide6 desktop window (`launch_desktop_app`) connecting directly to your training loop or exported benchmarks.

## 1. Setting up the Architecture & Live Diagnostics

`LiveTrainingDiagnostics` hooks directly into your model's forward and backward passes to capture per-layer gradient/activation statistics, detect dead neurons, and identify gradient anomalies without slowing down training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from graphyco import LiveTrainingDiagnostics

# Define architecture
model = nn.Sequential(
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 10)
)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Initialize live diagnostics
diagnostics = LiveTrainingDiagnostics(model)

print(f"Model: {model.__class__.__name__} with {sum(p.numel() for p in model.parameters()):,} parameters")
print("LiveTrainingDiagnostics ready.")

Model: Sequential with 10,666 parameters
LiveTrainingDiagnostics ready.


## 2. Training Loop with Live Health Monitoring

Wrap training iterations in `diagnostics.observe_step(step)`. The diagnostics monitor automatically records tensor statistics, step latency, and anomalous behavior.

In [2]:
# Run training loop and stream telemetry
for step in range(30):
    x = torch.randn(16, 128, requires_grad=True)
    y = torch.randint(0, 10, (16,))

    with diagnostics.observe_step(step):
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        diagnostics.current_loss = loss.item()
        loss.backward()
        optimizer.step()

    if step % 10 == 0:
        snap = diagnostics.get_snapshot()
        print(f"Step {step:2d} | Loss: {loss.item():.4f} | Health: {snap['health_status']} | Bottleneck: layer {snap['active_bottleneck_layer']}")

print("\nTraining complete.")

Step  0 | Loss: 2.3312 | Health: HEALTHY | Bottleneck: layer 4
Step 10 | Loss: 2.3388 | Health: WARNING: High Dead Neuron Ratio | Bottleneck: layer 4
Step 20 | Loss: 2.3099 | Health: HEALTHY | Bottleneck: layer 4

Training complete.


## 3. Real-Time Health Snapshot Inspection

Inspect gradient RMS, activation RMS, and dead neuron percentages across all layers.

In [3]:
snapshot = diagnostics.get_snapshot()

print(f"Health Status     : {snapshot['health_status']}")
print(f"Active Bottleneck : layer {snapshot['active_bottleneck_layer']} (G_max={snapshot['gmax']:.2f})")
print(f"Avg Step Time     : {snapshot['avg_step_time_ms']:.2f} ms")
print(f"Alerts Recorded   : {len(snapshot['alerts'])}")

# Tabulate layer metrics
rows = []
for layer_name, stats in sorted(snapshot["layers"].items()):
    rows.append({
        "Layer": layer_name,
        "Grad RMS": f"{stats['latest_grad_rms']:.6f}",
        "Act RMS": f"{stats['latest_act_rms']:.6f}",
        "Dead Neuron %": f"{stats['dead_neuron_ratio'] * 100:.1f}%"
    })
pd.DataFrame(rows)

Health Status     : WARNING: High Dead Neuron Ratio
Active Bottleneck : layer 4 (G_max=2.90)
Avg Step Time     : 1.49 ms
Alerts Recorded   : 5


,Layer,Grad RMS,Act RMS,Dead Neuron %
0,0,0.001209,0.572727,0.0%
1,1,0.001777,0.395486,52.8%
2,2,0.004392,0.224487,0.0%
3,3,0.006171,0.162890,47.7%
4,4,0.018728,0.141007,0.0%


## 4. In-Notebook Interactive Visualization (`visualize_inline`)

You can display the complete, interactive graph and flow telemetry dashboard directly inside the notebook cell without launching an external desktop window.

In [4]:
from graphyco.visualizer.gui import visualize_inline

# Renders the interactive graph dashboard directly into this cell
visualize_inline(model, inputs=torch.randn(8, 128), steps=5)

Invariant / Flow Property,Sym,Value
Topological Bottleneck,B(G),-
Gradient Concentration,Gmax,-
Bottleneck Node,v*,-
Connectivity Coherence,C(G),-
Structural Resilience,R(G),-
"Spearman ρ(a,g)",ρAG,-
"Pearson r(a,g)",rAG,-
Temporal CV (Mean),CV,-
Type I / II / III / IV,NI-IV,-
Metric,Sym,Current Step


## 5. Launching the Real-Time Desktop GUI (`PySide6`)

For an independent desktop window updating live every 200 ms during training, launch `VisualizerDesktopApp`.

> **Note:** In an interactive Jupyter Notebook with a display server, enable non-blocking Qt event integration using `%gui qt`. For standalone scripts, use `app.exec()` or `python run_live_demo.py`.

```python
# 1. Enable non-blocking Qt event loop for Jupyter
%gui qt

import threading
import time
from PySide6.QtWidgets import QApplication
from PySide6.QtCore import QCoreApplication, Qt
from graphyco.visualizer.gui import launch_desktop_app

# Ensure OpenGL context sharing
if not QApplication.instance():
    QCoreApplication.setAttribute(Qt.ApplicationAttribute.AA_ShareOpenGLContexts)

app = QApplication.instance() or QApplication([])
window = launch_desktop_app(diagnostics=diagnostics)

# 2. Background training loop streaming to the GUI
def training_thread():
    for step in range(100):
        x = torch.randn(16, 128, requires_grad=True)
        y = torch.randint(0, 10, (16,))
        with diagnostics.observe_step(step):
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            diagnostics.current_loss = loss.item()
            loss.backward()
            optimizer.step()
        time.sleep(0.1)
    print("Training finished.")

thread = threading.Thread(target=training_thread, daemon=True)
thread.start()
```

## 6. Standalone Live Demo & CLI Visualizer

You can also run the pre-configured live training demo or inspect exported benchmarks directly from the command line:

```bash
# 1. Run live training demo with background PyTorch worker (MLP, ResNet, UNet, CNN, etc.)
python run_live_demo.py --arch mlp

# 2. Inspect an exported benchmark JSON in the desktop visualizer
python -m graphyco.visualizer --json my_model_benchmark.json

# 3. Query bottleneck analysis from the terminal without starting the GUI
python -m graphyco.visualizer --json my_model_benchmark.json --query bottlenecks
```